# Graph Fraud Detection — SPCS Deployment + Snowflake Gateway

Deploys `FRAUD_RING_MODEL` to SPCS and creates a **Snowflake Gateway** for stable routing.

| Step | Description |
|------|-------------|
| 1 | Deploy V1 (prod) as `GRAPH_FRAUD_SVC_V1` |
| 2 | Deploy V2 (candidate) as `GRAPH_FRAUD_SVC_V2` |
| 3 | Create Gateway — 100% → V1 |
| 4 | Canary promotion 90/10 → 50/50 → 0/100 |
| 5 | Get stable Gateway URL (used by simulator) |

The Gateway provides a **permanent hostname** that survives service recreation,
enabling zero-downtime model upgrades via `ALTER GATEWAY`.

In [1]:
import os, time
import snowflake.connector
from snowflake.snowpark import Session
from snowflake.ml.registry import Registry

DATABASE     = 'GRAPH_FRAUD_DEMO'
SCHEMA       = 'MODELS'
WAREHOUSE    = 'GRAPH_FRAUD_WH'
COMPUTE_POOL = 'GRAPH_FRAUD_INFERENCE_POOL'
MODEL_NAME   = 'FRAUD_RING_MODEL'
GATEWAY_NAME = 'GRAPH_FRAUD_GATEWAY'
SVC_V1       = 'GRAPH_FRAUD_SVC_V1'
SVC_V2       = 'GRAPH_FRAUD_SVC_V2'

# Load PAT from Snowflake config
import tomllib, pathlib
#_sf_config = tomllib.load(open(pathlib.Path.home() / '.snowflake' / 'config.toml', 'rb'))
print(f'SNOWFLAKE_PAT loaded ({len(os.environ["SNOWFLAKE_PAT"])} chars)')

In [2]:
try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    conn = snowflake.connector.connect(connection_name='parker_demo')
    session = Session.builder.configs({'connection': conn}).create()

session.sql('USE ROLE ACCOUNTADMIN').collect()
session.sql(f'USE DATABASE {DATABASE}').collect()
session.sql(f'USE SCHEMA {DATABASE}.{SCHEMA}').collect()
session.sql(f'USE WAREHOUSE {WAREHOUSE}').collect()
print(f'Connected: {session.get_current_account()}')

## Step 1 & 2: Create Compute Pool & Deploy Services

Each model version gets its own SPCS service. The Gateway routes between them.

In [3]:
# Drop existing services and gateway to allow clean re-creation
for svc in ['GRAPH_FRAUD_SVC_V1', 'GRAPH_FRAUD_SVC_V2']:
    try: session.sql(f'DROP SERVICE IF EXISTS {DATABASE}.{SCHEMA}.{svc}').collect()
    except: pass
try: session.sql(f'DROP GATEWAY IF EXISTS {DATABASE}.{SCHEMA}.{GATEWAY_NAME}').collect()
except: pass
print('Cleaned up existing services and gateway')

In [4]:

session.sql(f"""
    CREATE COMPUTE POOL IF NOT EXISTS {COMPUTE_POOL}
      MIN_NODES = 1 MAX_NODES = 3
      INSTANCE_FAMILY = CPU_X64_S
""").collect()
print(f'Compute pool ready: {COMPUTE_POOL}')

reg = Registry(session=session, database_name=DATABASE, schema_name=SCHEMA)
model_ref = reg.get_model(MODEL_NAME)

for version, svc_name in [('V1', SVC_V1), ('V2', SVC_V2)]:
    mv = model_ref.version(version)
    try:
        mv.create_service(
            service_name=svc_name,
            service_compute_pool=COMPUTE_POOL,
            image_build_compute_pool=COMPUTE_POOL,
            ingress_enabled=True,
            max_instances=1,
        )
        print(f'  Service creation initiated: {svc_name} ({version})')
    except Exception as e:
        if 'already exists' in str(e).lower():
            print(f'  Service already exists: {svc_name}')
        else:
            raise

In [5]:
print('Waiting for services to reach RUNNING (~5-10 min for image build)...')
for svc_name in [SVC_V1, SVC_V2]:
    for i in range(30):
        rows = session.sql(
            f"SHOW SERVICES LIKE '{svc_name}' IN SCHEMA {DATABASE}.{SCHEMA}"
        ).collect()
        svc_status = rows[0].as_dict().get('status', 'UNKNOWN') if rows else 'NOT_FOUND'
        print(f'  {svc_name}: {svc_status}')
        if svc_status == 'RUNNING':
            break
        time.sleep(30)
    else:
        print(f'  WARNING: {svc_name} not RUNNING after 15 min')

## Step 3: Create Snowflake Gateway

Provides a **permanent stable URL** across upgrades.
Initially 100% → V1.

Required privileges:
- `CREATE GATEWAY` on schema
- `ALL_ENDPOINTS_USAGE` on each service
- `BIND SERVICE ENDPOINT` on account

In [6]:
session.sql('GRANT BIND SERVICE ENDPOINT ON ACCOUNT TO ROLE ACCOUNTADMIN').collect()

for svc_name in [SVC_V1, SVC_V2]:
    try:
        session.sql(
            f'GRANT SERVICE ROLE {DATABASE}.{SCHEMA}.{svc_name}!ALL_ENDPOINTS_USAGE TO ROLE ACCOUNTADMIN'
        ).collect()
    except Exception as e:
        print(f'Grant note ({svc_name}): {e}')

session.sql(f"""
CREATE OR REPLACE GATEWAY {DATABASE}.{SCHEMA}.{GATEWAY_NAME}
  FROM SPECIFICATION $$
    spec:
      type: traffic_split
      split_type: custom
      targets:
        - type: endpoint
          value: {DATABASE}.{SCHEMA}.{SVC_V1}!inference
          weight: 100
        - type: endpoint
          value: {DATABASE}.{SCHEMA}.{SVC_V2}!inference
          weight: 0
  $$
""").collect()

print(f'Gateway created: {GATEWAY_NAME}')
print('  V1: 100% traffic (production)')
print('  V2:   0% traffic (standing by)')

In [7]:
def get_gateway_url(max_wait_min=8):
    """Resolve the Gateway ingress URL, waiting out endpoint provisioning."""
    deadline = time.time() + max_wait_min * 60
    while True:
        rows = session.sql(f'DESC GATEWAY {DATABASE}.{SCHEMA}.{GATEWAY_NAME}').collect()
        raw = ''
        for row in rows:
            raw = (row.as_dict().get('ingress_url') or '').strip()
            if raw:
                break

        provisioning = (not raw) or ('provisioning' in raw.lower()) or (' ' in raw) or ('.' not in raw)
        if not provisioning:
            url = raw if raw.startswith('http') else f'https://{raw}'
            return url

        remaining = int(deadline - time.time())
        if remaining <= 0:
            print(f'Endpoint still provisioning after {max_wait_min} min. Raw value: "{raw}"')
            print('Re-run this cell shortly — provisioning a gateway hostname is a one-time ~2-5 min step.')
            return None
        print(f'  Endpoint provisioning... retrying in 30s (up to {remaining}s left)')
        time.sleep(30)

GATEWAY_URL = get_gateway_url()

if GATEWAY_URL:
    print(f'Gateway stable URL : {GATEWAY_URL}')
    print(f'Inference endpoint : {GATEWAY_URL}/predict-proba')
    print()
    print('Save GATEWAY_URL — this is what simulator/predict.py calls:')
    print(f'  export GRAPH_FRAUD_GATEWAY_URL="{GATEWAY_URL}"')

## Step 4: Canary Promotion

Shift traffic gradually: 90/10 → 50/50 → 0/100.
Clients always call the **same Gateway URL** — no config changes.

In [8]:
def shift_gateway(v1_pct, v2_pct):
    session.sql(f"""
    ALTER GATEWAY {DATABASE}.{SCHEMA}.{GATEWAY_NAME}
      FROM SPECIFICATION $$
        spec:
          type: traffic_split
          split_type: custom
          targets:
            - type: endpoint
              value: {DATABASE}.{SCHEMA}.{SVC_V1}!inference
              weight: {v1_pct}
            - type: endpoint
              value: {DATABASE}.{SCHEMA}.{SVC_V2}!inference
              weight: {v2_pct}
      $$
    """).collect()
    print(f'  Traffic: V1={v1_pct}%  V2={v2_pct}%')

print('=== Canary Promotion Demo ===')
print('Step 1: Canary (10% to V2)')
shift_gateway(90, 10)
time.sleep(3)
print('Step 2: 50/50 split')
shift_gateway(50, 50)
time.sleep(3)
print('Step 3: Full cut-over to V2')
shift_gateway(0, 100)
print()
print('V2 now serves 100% of traffic via the same Gateway URL.')
print(f'Suspend V1 when confident:')
print(f'  ALTER SERVICE {DATABASE}.{SCHEMA}.{SVC_V1} SUSPEND;')

## Step 5: Rollback

If V2 shows degraded fraud metrics, restore V1 instantly with zero client impact.

In [9]:
# To roll back to V1 instantly (clients see no URL change):
# shift_gateway(100, 0)
print('Rollback: shift_gateway(100, 0)')
print()
print('Proceed to 04_monitor.ipynb')